In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VisionEncoder(nn.Module):
    def __init__(
        self,
        img_size=64,
        patch_size=16,
        in_chans=3,
        embed_dim=256,
        depth=6,
        num_heads=1,
    ):
        super(VisionEncoder, self).__init__()
        assert img_size % patch_size == 0, "Image size must be divisible by patch size."
        self.num_patches = (img_size // patch_size) ** 2
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.depth = depth
        self.grid_size = img_size // patch_size  # Number of patches along one dimension

        # Patch embedding layer
        self.patch_embed = nn.Conv2d(
            in_channels=in_chans,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
        )

        pos_embed = self._get_2d_sincos_pos_embed(embed_dim, self.grid_size)
        pos_embed = pos_embed.unsqueeze(0)  # Shape: (1, num_patches, embed_dim)
        self.register_buffer('pos_embed', pos_embed) 
        # Note that trainable embeddings work better, this is just for compute savings.

        # Transformer encoder layers as a ModuleList
        self.encoder_layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=embed_dim, nhead=num_heads, batch_first=True,
                dim_feedforward=embed_dim * 4, dropout=0
            ) for _ in range(depth)
        ])


    def forward(self, x):
        """
        x: Input tensor of shape (batch_size, 3, img_size, img_size)
        Returns:
            weighted_output: Tensor of shape (batch_size, num_patches, embed_dim)
        """
        batch_size = x.size(0)

        # Patch embedding
        x = self.patch_embed(x)  # Shape: (batch_size, embed_dim, H_p, W_p)
        x = x.flatten(2)         # Shape: (batch_size, embed_dim, num_patches)
        x = x.transpose(1, 2)    # Shape: (batch_size, num_patches, embed_dim)

        # Add fixed 2D sine-cosine positional embeddings
        x = x + self.pos_embed   # Shape: (batch_size, num_patches, embed_dim)

        # Collect outputs from all layers
        layer_outputs = []
        for layer in self.encoder_layers:
            x = layer(x)  # Shape: (batch_size, num_patches, embed_dim)
            layer_outputs.append(x.unsqueeze(1))  # Shape: (batch_size, 1, num_patches, embed_dim)

        # Stack all layer outputs: Shape (batch_size, depth, num_patches, embed_dim)
        stacked_outputs = torch.cat(layer_outputs, dim=1)

        return stacked_outputs

    def _get_2d_sincos_pos_embed(self, embed_dim, grid_size):
        """
        Generate 2D sine-cosine positional embeddings.

        Args:
            embed_dim (int): Dimension of the embeddings. Must be divisible by 2 * grid_size.
            grid_size (int): Number of patches along one dimension.

        Returns:
            torch.Tensor: Positional embeddings of shape (num_patches, embed_dim)
        """
        assert embed_dim % 2 == 0, "Embed dimension must be even for sine and cosine."

        # Compute the number of patches per axis
        grid_h = torch.arange(grid_size, dtype=torch.float32)
        grid_w = torch.arange(grid_size, dtype=torch.float32)
        grid_h, grid_w = torch.meshgrid(grid_h, grid_w, indexing='ij')  # Shape: (grid_size, grid_size)

        grid_h = grid_h.flatten()  # Shape: (num_patches,)
        grid_w = grid_w.flatten()  # Shape: (num_patches,)

        # Compute sine and cosine embeddings for height and width
        pos_h = self._get_1d_sincos_pos_embed(embed_dim // 2, grid_h)  # Shape: (num_patches, embed_dim // 2)
        pos_w = self._get_1d_sincos_pos_embed(embed_dim // 2, grid_w)  # Shape: (num_patches, embed_dim // 2)

        # Concatenate height and width embeddings
        pos = torch.cat([pos_h, pos_w], dim=1)  # Shape: (num_patches, embed_dim)

        return pos

    def _get_1d_sincos_pos_embed(self, embed_dim, pos):
        """
        Generate 1D sine-cosine positional embeddings.

        Args:
            embed_dim (int): Dimension of the embeddings. Must be even.
            pos (torch.Tensor): Positions (integer indices).

        Returns:
            torch.Tensor: Positional embeddings of shape (num_patches, embed_dim)
        """
        assert embed_dim % 2 == 0, "Embed dimension must be even for sine and cosine."

        omega = torch.arange(embed_dim // 2, dtype=torch.float32) / (embed_dim / 2)
        omega = 1. / (10000 ** omega)  # Shape: (embed_dim // 2,)

        pos = pos[:, None] * omega[None, :]  # Shape: (num_patches, embed_dim // 2)
        emb_sin = torch.sin(pos)             # Shape: (num_patches, embed_dim // 2)
        emb_cos = torch.cos(pos)             # Shape: (num_patches, embed_dim // 2)
        emb = torch.cat([emb_sin, emb_cos], dim=1)  # Shape: (num_patches, embed_dim)

        return emb

In [ ]:
!pip install kornia
import torch
import torch.nn as nn
import kornia.augmentation as K

# You can actually do paired augmentation with pytorch if tou redefine the classes, the choice of Kornia just let me get the forward parameters. 
# It's VERY important that the same augmentations are applied to your observations so you can capture a better 'delta'.

class PairedRandomAugment(nn.Module):
    """
    Ensures that for each sample i in the batch,
    the same random transform is applied to (x1[i], x2[i]).
    The transform parameters are drawn once per-batch element, then reused.
    """
    def __init__(self):
        super().__init__()
        self.transforms = K.AugmentationSequential(
            K.RandomHorizontalFlip(p=0.5),
            K.RandomResizedCrop(
                size=(64, 64),
                scale=(0.8, 1.0),
                ratio=(0.9, 1.1),
                p=0.5
            ),
            K.ColorJitter(
                brightness=0.4,
                contrast=0.4,
                saturation=0.3,
                hue=0.1,
                p=0.8
            ),
            K.RandomGrayscale(p=0.2),
            K.RandomGaussianBlur(kernel_size=(3, 3), sigma=(0.1, 2.0), p=0.2),
            K.RandomAffine(
                degrees=15,
                translate=(0.1, 0.1),
                scale=(0.9, 1.1),
                shear=(-10, 10),
                p=0.3
            ),
            K.RandomPerspective(distortion_scale=0.2, p=0.2),
            K.RandomErasing(
                scale=(0.02, 0.15),
                ratio=(0.3, 3.3),
                same_on_batch=False,
                p=0.3
            ),
            data_keys=["input"],  # Ensures the same augmentation params for each item
        )

    def forward(self, x1: torch.Tensor, x2: torch.Tensor):
        """
        Args:
            x1, x2: Float images in [0, 1], shape [B, C, H, W].
        Returns:
            x1_out, x2_out: Transformed images (same random transform for x1[i], x2[i]).
        """
        # 1) Generate the random augmentation parameters for each sample
        params = self.transforms.forward_parameters(x1.shape)

        # 2) Apply the same transform to x1, x2
        x1_out = self.transforms(x1, params=params)
        x2_out = self.transforms(x2, params=params)
        return x1_out, x2_out



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 68.2 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy

##############################################################################
# 1) The DoubleCrossDecoder layer and stack
##############################################################################

class DoubleCrossDecoderLayer(nn.Module):
    """
    A Transformer-style decoder layer that attends to two separate memories:
      - memory_state (previous-state embeddings)
      - memory_action (action embeddings)
    """
    def __init__(self, d_model, nhead=4, dim_feedforward=1024, dropout=0.1):
        super().__init__()
        # # 1) Self-attention (on decoder input)
        # self.self_attn = nn.MultiheadAttention(
        #     embed_dim=d_model,
        #     num_heads=nhead,
        #     dropout=dropout,
        #     batch_first=True,
        # )
        # 2) Cross-attention to the "state" memory
        self.cross_attn_state = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            dropout=dropout,
            batch_first=True,
        )
        # 3) Cross-attention to the "action" memory
        self.cross_attn_action = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            dropout=dropout,
            batch_first=True,
        )

        # Feed-forward block
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        # LayerNorms
        # self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.norm4 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)
        self.activation = F.relu

    def forward(self, tgt, memory_state, memory_action):
        """
        tgt          : [B, T_dec, d_model]  (decoder input)
        memory_state : [B, T_state, d_model]
        memory_action: [B, T_action, d_model]
        """
        # # 1) Self-attention
        # x = self.self_attn(tgt, tgt, tgt)[0]
        # tgt = self.norm1(tgt + x)

        # 2) Cross-attention to 'memory_state'
        x = self.cross_attn_state(tgt, memory_state, memory_state)[0]
        tgt = self.norm2(tgt + x)

        # 3) Cross-attention to 'memory_action'
        x = self.cross_attn_action(tgt, memory_action, memory_action)[0]
        tgt = self.norm3(tgt + x)

        # 4) Feed-forward
        x = self.linear2(self.dropout(self.activation(self.linear1(tgt))))
        tgt = self.norm4(tgt + x)

        return tgt


class DoubleCrossDecoder(nn.Module):
    """
    A stack of DoubleCrossDecoderLayers, forming a multi-layer double-cross-attention decoder.
    """
    def __init__(self, d_model, nhead=4, num_layers=2, dim_feedforward=1024, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            DoubleCrossDecoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, tgt, memory_state, memory_action):
        """
        Pass through each DoubleCrossDecoderLayer in turn.
        """
        output = tgt
        for layer in self.layers:
            output = layer(output, memory_state, memory_action)
        return output

class PairedRandomAugmentWM(nn.Module):
    """
    Ensures that for each sample i in the batch, the same random transform
    is applied to (x1[i], x2[i]). The transform parameters are drawn per sample.
    """
    def __init__(self):
        super().__init__()
        self.transforms = K.AugmentationSequential(
            K.RandomResizedCrop(
                size=(64, 64),
                scale=(0.2, 0.5),
                ratio=(0.9, 1.1),
                p=1
            ),
            data_keys=["input"],
        )

    def forward(self, x1: torch.Tensor, x2: torch.Tensor):
        params = self.transforms.forward_parameters(x1.shape)
        x1_out = self.transforms(x1, params=params)
        x2_out = self.transforms(x2, params=params)
        return x1_out, x2_out

class WorldModel(nn.Module):
    def __init__(
        self,
        embed_dim=256,        # E
        action_dim=128,
        img_size=64,
        patch_size=8,
        wm_num_heads=1,
        wm_depth=1,
        ve_depth=3,
    ):
        """
        WorldModel that:
         - Uses a VisionEncoder (plus EMA copy) for "state" memory.
         - Uses an ActionEncoder (simple linear) for "action" memory.
         - Runs a custom DoubleCrossDecoder to attend to both memories.
        """
        super().__init__()
        self.embed_dim = embed_dim
        self.action_dim = action_dim
        self.ve_depth = ve_depth  # #layers in the VisionEncoder
        self.img_size = img_size
        self.patch_size = patch_size

        ######################################################################
        # 1) Vision Encoder + EMA Copy
        ######################################################################
        self.encoder = VisionEncoder(
            img_size=img_size,
            patch_size=patch_size,
            embed_dim=embed_dim,
            depth=ve_depth,
            num_heads=4,
        )
        self.ema_encoder = copy.deepcopy(self.encoder)
        for param in self.ema_encoder.parameters():
            param.requires_grad = False

        self.paired_augment = PairedRandomAugmentWM()

        # Number of patches
        self.num_patches = self.encoder.num_patches  # P
        # D = self.ve_depth
        # E = self.embed_dim
        # d_model = D * E
        self.d_model = self.ve_depth * self.embed_dim

        ######################################################################
        # 2) Action Encoder
        ######################################################################
        # We'll encode the action into the same d_model dimension.
        # If you prefer, make a bigger MLP; here we do a single Linear for simplicity.
        self.action_encoder = nn.Linear(action_dim, self.d_model)

        ######################################################################
        # 3) Double-Cross-Attention Decoder
        ######################################################################
        self.decoder = DoubleCrossDecoder(
            d_model=self.d_model,
            nhead=wm_num_heads,
            num_layers=wm_depth,            # how many DoubleCrossDecoderLayer's
            dim_feedforward=self.d_model, # or set as you like
            dropout=0.0,
        )

        # We'll use "mask tokens" as the decoder input queries:
        self.num_mask_tokens = self.num_patches
        self.mask_tokens = nn.Parameter(torch.randn(1, self.num_mask_tokens, self.d_model) * 0.02)

    def forward(self, before_obs, action):
        """
        1) Encode 'before_obs' with EMA => memory_state of shape [B, P, d_model].
        2) Encode 'action' => memory_action of shape [B, 1, d_model].
        3) Pass a set of learned "mask tokens" through the double-cross decoder
           that cross-attends to memory_state + memory_action.
        4) Reshape the decoder output => [B, D, P, E]
        """
        B = before_obs.shape[0]
        D = self.ve_depth
        P = self.num_patches
        E = self.embed_dim

        ######################################################################
        # 1) Encode the previous state with the EMA VisionEncoder
        ######################################################################

        with torch.no_grad():
            # Get EMA encoder output: shape [B, D, P, E]
            state_embed = self.ema_encoder(before_obs).detach()

        # Flatten (D, E) into a single dimension => d_model = D*E
        # state_embed => [B, P, d_model]
        state_embed = state_embed.permute(0, 2, 1, 3).contiguous()
        state_embed = state_embed.view(B, P, D*E)

        ######################################################################
        # 2) Encode the action => memory_action: [B, num_action_embeds, d_model]
        ######################################################################
        memory_action = self.action_encoder(action)

        ######################################################################
        # 3) Prepare decoder input: mask tokens => shape [B, P, d_model]
        ######################################################################
        # We have self.mask_tokens => [1, P, d_model]
        # broadcast to batch dimension => [B, P, d_model]
        decoder_input = self.mask_tokens.expand(B, -1, -1)

        # Double cross attention decode:
        #   - self-attn on the mask tokens
        #   - cross-attn to memory_state
        #   - cross-attn to memory_action
        out = self.decoder(decoder_input, state_embed, memory_action)
        # out => [B, P, d_model]

        ######################################################################
        # 4) Reshape => [B, D, P, E]
        ######################################################################
        out = out.view(B, P, D, E).permute(0, 2, 1, 3).contiguous()

        return out

    def label(self, batch, num_local_views=4):
        """
        Computes two sets of prediction losses:
          1. Global view loss: using the original (unaltered) images.
          2. Local view loss: using paired augmented (cropped/resized) images.

        The augmentation is vectorized: for each batch element,
        we create 'num_local_views' copies, apply the same random crop transform
        (using PairedRandomAugment) to both the before_obs and target_obs, and then
        predict using the same forward pass.

        Returns:
            total_loss : combined loss (global + local), shape [D] (per depth)
            loss_global: loss from global predictions, shape [D]
            loss_local : loss from local (augmented) predictions, shape [D]
        """
        la = batch["la"]  # shape: [B, num_toks, action_dim]
        before_obs = batch["obs"][:, -2]  # global view: [B, C, H, W]
        target_obs = batch["obs"][:, -1]    # global view: [B, C, H, W]
        B, C, H, W = before_obs.shape

        ######################################################################
        # Global Prediction and Loss (unchanged)
        ######################################################################
        with torch.no_grad():
            # EMA encoder on target_obs gives target embeddings of shape [B, D, P, E]
            target_embed_global = self.ema_encoder(target_obs)
        # Forward pass on the unaltered before_obs: shape [B, D, P, E]
        pred_global = self(before_obs, la)
        # Compute loss (e.g. MSE) per depth over all samples and patches
        loss_global = F.mse_loss(pred_global, target_embed_global, reduction='none')
        loss_global = loss_global.mean(dim=(0, 2, 3))  # shape: [D]

        ######################################################################
        # Local Prediction with Vectorized Paired Augmentation
        ######################################################################
        L = num_local_views
        # Expand before_obs and target_obs to shape [B, L, C, H, W]
        before_obs_local = before_obs.unsqueeze(1).expand(B, L, C, H, W).reshape(B * L, C, H, W)
        target_obs_local = target_obs.unsqueeze(1).expand(B, L, C, H, W).reshape(B * L, C, H, W)

        # Instantiate your paired augmentation module (ensure it’s on the correct device)

        before_obs_aug, target_obs_aug = self.paired_augment(before_obs_local, target_obs_local)

        # Expand la accordingly: from [B, num_toks, action_dim] to [B, L, num_toks, action_dim],
        # then reshape to [B * L, num_toks, action_dim]
        la_local = la.unsqueeze(1).expand(B, L, la.shape[1], la.shape[2]).reshape(B * L, la.shape[1], la.shape[2])

        # Forward pass on the augmented before_obs
        pred_local = self(before_obs_aug, la_local)
        with torch.no_grad():
            target_embed_local = self.ema_encoder(target_obs_aug)
        loss_local = F.mse_loss(pred_local, target_embed_local, reduction='none')
        loss_local = loss_local.mean(dim=(0, 2, 3))  # shape: [D]
        # I do this because I was trying to do depth weighting at some point, it's not necessary. Trained gating collapses into just focusing on one of the depths (often the earliest.)

        return loss_global, loss_local


    @torch.no_grad()
    def update_ema(self, momentum=0.9):
        """
        EMA update for the online encoder -> target/EMA encoder.
        """
        for ema_param, param in zip(self.ema_encoder.parameters(), self.encoder.parameters()):
            ema_param.data.mul_(momentum).add_(param.data * (1 - momentum))

    def get_encoder(self):
        return self.encoder

    def get_ema_encoder(self):
        return self.ema_encoder


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tensordict import TensorDict


class IDM(nn.Module):
    def __init__(
        self,
        obs_shape,
        embed_dim,
        action_dim,
        encoder,
        ema_encoder,
        pool_size=(8, 8),
        mlp_hidden_dim=512,     # no longer used, but kept for compatibility
        depth_embedder=None,
        num_action_tokens=6,
    ):
        super().__init__()

        # Store references
        self.encoder = encoder
        self.ema_encoder = ema_encoder
        self.ema_encoder.requires_grad_(False)

        self.depth = self.encoder.depth           # number of layers in the VisionEncoder
        self.num_patches = self.encoder.num_patches
        self.embed_dim = self.encoder.embed_dim   # E

        # Our model dimension (d_model) is set to depth * embed_dim.
        self.d_model = self.depth * self.embed_dim

        # Update temporal embeddings to have shape [1, 1, d_model] so they broadcast over [B, num_patches, d_model].
        self.before_embedding = nn.Parameter(torch.randn(1, 1, self.d_model) * 0.02)
        self.after_embedding = nn.Parameter(torch.randn(1, 1, self.d_model) * 0.02)

        # An embedding mixer that now takes in a d_model–dimensional input.
        self.embed_mix = nn.Sequential(
            nn.Linear(self.d_model, self.d_model),
            nn.ReLU(),
            nn.Linear(self.d_model, self.d_model)
        )

        # Transformer encoder: we use only the encoder part to mix the tokens.
        self.transformer = nn.Transformer(
            d_model=self.d_model,
            nhead=8,
            num_encoder_layers=2,
            num_decoder_layers=2,  # not used in this design
            dim_feedforward=self.d_model * 4,
            dropout=0.0,
            activation='relu',
            batch_first=True
        )

        # Separate transformer decoders for before and after observations.
        decoder_layer_before = nn.TransformerDecoderLayer(
            d_model=self.d_model,
            nhead=8,
            dim_feedforward=self.d_model * 4,
            dropout=0.0,
            activation='relu',
            batch_first=True
        )
        decoder_layer_after = nn.TransformerDecoderLayer(
            d_model=self.d_model,
            nhead=8,
            dim_feedforward=self.d_model * 4,
            dropout=0.0,
            activation='relu',
            batch_first=True
        )
        self.decoder_before = nn.TransformerDecoder(decoder_layer_before, num_layers=2)
        self.decoder_after  = nn.TransformerDecoder(decoder_layer_after, num_layers=2)

        # Learnable action tokens for the decoders, shape: [1, num_action_tokens, d_model]
        self.num_action_tokens = num_action_tokens
        self.action_tokens = nn.Parameter(torch.randn(1, num_action_tokens, self.d_model) * 0.02)

        # Final head to project decoder output to the desired action dimension.
        self.action_head = nn.Linear(self.d_model, action_dim)

        self.aug_fn = PairedRandomAugment()

    def forward(self, x_before, x_after):
        batch_size = x_before.size(0)

        # 1) Encode: obtain tensors of shape [B, depth, num_patches, embed_dim]
        before_enc = self.encoder(x_before)
        after_enc  = self.encoder(x_after)

        # 2) Reshape: Permute and flatten the depth and embed_dim dimensions to get [B, num_patches, d_model]
        B, D, P, E = before_enc.shape  # D = depth, E = embed_dim, P = num_patches
        before = before_enc.permute(0, 2, 1, 3).reshape(B, P, D * E)
        after  = after_enc.permute(0, 2, 1, 3).reshape(B, P, D * E)

        # 3) Apply embed_mix which now mixes all channels (d_model dimension)
        before_embed = self.embed_mix(before)
        after_embed  = self.embed_mix(after)

        # 4) Add temporal embeddings
        before_embed = before_embed + self.before_embedding  # [B, num_patches, d_model]
        after_embed  = after_embed + self.after_embedding       # [B, num_patches, d_model]

        # 5) Concatenate before and after along the patch dimension => [B, 2P, d_model]
        merged = torch.cat([before_embed, after_embed], dim=1)

        # 6) Pass through transformer encoder: output shape [B, 2P, d_model]
        encoder_out = self.transformer.encoder(merged)

        # 7) Split the encoder output into before and after tokens along the sequence dimension.
        before_mem = encoder_out[:, :self.num_patches, :]  # [B, P, d_model]
        after_mem  = encoder_out[:, self.num_patches:, :]   # [B, P, d_model]

        # 8) Prepare and run decoders using learnable action tokens.
        action_tokens = self.action_tokens.expand(batch_size, -1, -1)
        action_tokens = self.decoder_before(tgt=action_tokens, memory=before_mem)
        action_tokens = self.decoder_after(tgt=action_tokens, memory=after_mem)

        # 9) Combine outputs and project to action_dim.
        action_embedding = self.action_head(action_tokens)  # [B, num_action_tokens, action_dim]
        action_embeds_flat = action_embedding.reshape(batch_size, -1)

        # Clean up (optional)
        del before_enc, after_enc, before, after, merged, encoder_out, action_tokens
        torch.cuda.empty_cache()

        return TensorDict(
            {
                "la": action_embedding,      # Unflattened: [B, num_action_tokens, action_dim]
                "la_flat": action_embeds_flat,  # Flattened: [B, num_action_tokens * action_dim]
            },
            batch_size=batch_size,
        )


    def label(self, batch):
        """
        Mutates the batch with predicted action embeddings,
        storing them under batch["la"].
        """
        x_before, x_after = batch["obs"][:, -2], batch["obs"][:, -1]
        with torch.no_grad():
            x_before, x_after = self.aug_fn(x_before, x_after)

        action_dict = self(x_before, x_after)
        batch.update(action_dict)
        del action_dict
        torch.cuda.empty_cache()

    @torch.no_grad()
    def label_chunked(
        self,
        data: TensorDict,
        chunksize: int = 128,
    ) -> TensorDict:
        """
        Example method for labeling large datasets in chunks
        (in inference mode).
        """
        self.encoder.eval()
        device = next(self.encoder.parameters()).device

        out_dicts = []

        def _label(data_batch: TensorDict):
            after_obs = normalize_obs(data_batch["obs"][:, -2].to(device))
            before_obs = normalize_obs(data_batch["obs"][:, -1].to(device))
            preds = self(before_obs, after_obs).to(data_batch.device)
            return preds

        for data_batch in data.split(chunksize):
            out = _label(data_batch)
            out_dicts.append(out.cpu())
            del out
            torch.cuda.empty_cache()

        action_dicts = torch.cat(out_dicts)
        data.update(action_dicts)
        del action_dicts
        torch.cuda.empty_cache()
        return data


In [ ]:
from tensordict import TensorDict
from torch import Tensor
from torch.utils.data import DataLoader, TensorDataset, random_split
import doy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

def obs_to_img(obs: Tensor) -> Tensor:
    return ((obs.permute(1, 2, 0) + 0.5) * 255).to(torch.uint8).numpy(force=True)


def create_decoder(la_dim, out_dim, device=DEVICE, hidden_sizes=(256, 128)):
    decoder = []
    in_size = la_dim
    for h in hidden_sizes:
        decoder.extend([nn.Linear(in_size, h), nn.ReLU()])
        in_size = h
    decoder.append(nn.Linear(h, out_dim))
    return nn.Sequential(*decoder).to(device)


def create_dynamics_models(
    model_cfg: ModelConfig, state_dicts: dict | None = None,
    embed_dim: int = 128,
) -> tuple[IDM, WorldModel]:
    obs_depth = 3
    idm_in_depth = obs_depth * 2  # ASSUMES NO ADD_TIME_HORIZON
    wm_in_depth = obs_depth * 1
    wm_out_depth = obs_depth


    wm = WorldModel(
        embed_dim=embed_dim,
        action_dim=model_cfg.la_dim,
        # obs_shape=(3, 64, 64),
    ).to(DEVICE)

    encoder = wm.get_encoder()

    ema_encoder = wm.get_ema_encoder()

    # depth_embedder = wm.get_depth_embedder()

    idm = IDM(
        (3, 64, 64),
        embed_dim,
        model_cfg.la_dim,
        encoder=encoder,
        ema_encoder=ema_encoder,
    ).to(DEVICE)

    if state_dicts is not None:
        idm.load_state_dict(state_dicts["idm"])
        wm.load_state_dict(state_dicts["wm"])

    return idm, wm



def eval_latent_repr(labeled_data: DataStager, idm: IDM, wm: WorldModel):
    batch = labeled_data.td_unfolded[:131072]
    actions = idm.label_chunked(batch).select("ta", "la_flat").to(DEVICE)
    TA_DIM = 15
    la_dim = actions["la_flat"].shape[-1]

    decoder = create_decoder(la_dim, TA_DIM)
    return train_decoder(data=actions, this_decoder=decoder, wm=wm)


def train_decoder(
    data: TensorDict,
    this_decoder: nn.Module,
    wm: WorldModel,
    epochs=3,
    bs=128,
):

    initial_params = [p.clone().detach() for p in this_decoder.parameters()]
    opt = torch.optim.AdamW(this_decoder.parameters(), lr=0.0001)
    logger = doy.Logger(use_wandb=False)

    train_data, test_data = data[: len(data) // 2], data[len(data) // 2 :]
    # train_dataloader = DataLoader(train_data, batch_size=bs, shuffle=True, drop_last=True)
    # print(train_dataloader)

    train_dataloader = DataLoader(
        train_data,  # type: ignore
        batch_size=bs,
        shuffle=False,
        collate_fn=lambda x: x,
    )

    test_dataloader = DataLoader(
        test_data,  # type: ignore
        batch_size=bs,
        shuffle=False,
        collate_fn=lambda x: x,
    )

    # test_dataloader = DataLoader(test_data, batch_size=bs, shuffle=False, drop_last=True)
    this_decoder.train()
    for i in range(10):  # make this larger
        for batch in train_dataloader:
            pred_ta = this_decoder(batch["la_flat"])
            ta = batch["ta"][:, -2]
            loss = F.cross_entropy(pred_ta, ta)

            opt.zero_grad()
            loss.backward()
            opt.step()

            logger(
                step=i,
                train_acc=(pred_ta.argmax(-1) == ta).float().mean(),
                train_loss=loss,
            )

    final_params = [p.clone().detach() for p in this_decoder.parameters()]
    param_changes = [torch.sum(torch.abs(f - i)).item() for f, i in zip(final_params, initial_params)]
    total_param_change = sum(param_changes)

    print(f"Total parameter change: {total_param_change}")
    this_decoder.eval()
    with torch.no_grad():
        test_losses = []
        test_accs = []
        for test_batch in test_dataloader:
            test_pred_ta = this_decoder(test_batch["la_flat"])
            test_ta = test_batch["ta"][:, -2]

            test_loss = F.cross_entropy(test_pred_ta, test_ta)
            test_acc = (test_pred_ta.argmax(-1) == test_ta).float().mean()

            test_losses.append(test_loss.item())
            test_accs.append(test_acc.item())

        logger(
            step=i,
            test_loss=np.mean(test_losses),
            test_acc=np.mean(test_accs),
        )

    metrics = dict(
        train_acc=np.mean(logger["train_acc"][-15:]),
        train_loss=np.mean(logger["train_loss"][-15:]),
        test_acc=logger["test_acc"][-1],
        test_loss=logger["test_loss"][-1],
    )

    return this_decoder, metrics





In [ ]:
default_cfg = get()
print_cfg(default_cfg)

env_name: starpilot                                                                                                
exp_name: starpilot_exp                                                                                            
stage_exp_name: null                                                                                               
model:                                                                                                             
  wm_scale: 24                                                                                                     
  idm_impala_scale: 4                                                                                              
  policy_impala_scale: 4                                                                                           
  vq:                                                                                                              
    enabled: true                                                                                                  
    num_codebooks: 2                                                                                               
    num_discrete_latents: 4                                                                                        
    emb_dim: 16                                                                                                    
    num_embs: 64                                                                                                   
    commitment_cost: 0.05                                                                                          
    decay: 0.999                                                                                                   
  la_dim: 128                                                                                                      
  ta_dim: 15                                                                                                       
stage1:                                                                                                            
  lr: 0.0003                                                                                                       
  bs: 128                                                                                                          
  steps: 50000                                                                                                     
stage2:                                                                                                            
  lr: 0.0002                                                                                                       
  bs: 128                                                                                                          
  steps: 60000                                                                                                     
stage3:                                                                                                            
  steps: 4000000                                                                                                   
  num_envs: 64                                                                                                     
  grad_accum_f: 1                                                                                                  
  num_steps: 64                                                                                                    
  num_minibatches: 8                                                                                               
  update_epochs: 3                                                                                                 
  ent_coef: 0.0025                                                                                                 
  lr: 0.01                                                                                                         
  anneal_lr: true                                       

In [ ]:
from doy import loop

"""

vq:
  enabled: true
  num_codebooks: 1
  num_discrete_latents: 4
  emb_dim: 32
  num_embs: 32
  commitment_cost: 0.2
  decay: 0.99409

"""
cfg = get()  # WE ONLY USE THE CONFIG's BS, STEPS AND LA_DIM
doy.print("[bold green]Running LAPO stage 1 (IDM/FDM training) with config:")
cfg.stage1.bs = 256
cfg.stage1.steps = 30000
cfg.model.la_dim = 128

Running LAPO stage 1 (IDM/FDM training) with config:

env_name: starpilot                                                                                                
exp_name: starpilot_exp                                                                                            
stage_exp_name: null                                                                                               
model:                                                                                                             
  wm_scale: 24                                                                                                     
  idm_impala_scale: 4                                                                                              
  policy_impala_scale: 4                                                                                           
  vq:                                                                                                              
    enabled: true                                                                                                  
    num_codebooks: 2                                                                                               
    num_discrete_latents: 4                                                                                        
    emb_dim: 16                                                                                                    
    num_embs: 64                                                                                                   
    commitment_cost: 0.05                                                                                          
    decay: 0.999                                                                                                   
  la_dim: 128                                                                                                      
  ta_dim: 15                                                                                                       
stage1:                                                                                                            
  lr: 0.0003                                                                                                       
  bs: 256                                                                                                          
  steps: 30000                                                                                                     
stage2:                                                                                                            
  lr: 0.0002                                                                                                       
  bs: 128                                                                                                          
  steps: 60000                                                                                                     
stage3:                                                                                                            
  steps: 4000000                                                                                                   
  num_envs: 64                                                                                                     
  grad_accum_f: 1                                                                                                  
  num_steps: 64                                                                                                    
  num_minibatches: 8                                                                                               
  update_epochs: 3                                                                                                 
  ent_coef: 0.0025                                                                                                 
  lr: 0.01                                                                                                         
  anneal_lr: true                                       

In [ ]:
import math
from torch.optim.lr_scheduler import LambdaLR

total_steps = cfg.stage1.steps

INITIAL_LR = 1e-7
PEAK_LR = 1e-4
FINAL_LR = 3e-5

def lr_lambda(step):
    warmup_steps = int(0.15 * total_steps)
    if step < warmup_steps:
        # Linear warmup from INITIAL_LR to PEAK_LR
        return (PEAK_LR - INITIAL_LR) / INITIAL_LR * (step / warmup_steps) + 1
    else:
        # Cosine decay from PEAK_LR to FINAL_LR
        progress = (step - warmup_steps) / (total_steps - warmup_steps)
        cosine_decay = 0.5 * (1 + math.cos(math.pi * progress + math.pi))
        return (PEAK_LR + (FINAL_LR - PEAK_LR) * cosine_decay) / INITIAL_LR


def get_unique_params(model1, model2):
    # note necessary, just make it into a set then a list again.
    params1 = set(model1.parameters())
    params2 = set(model2.parameters())
    return list(params1.union(params2))

In [ ]:
from doy import loop

# TODO, make VQ smaller support smaller latent action maybe, def make idm stronger

idm, wm = create_dynamics_models(cfg.model, embed_dim=256)

optimizer = torch.optim.AdamW(get_unique_params(idm, wm), lr=INITIAL_LR, weight_decay=0.05)
scheduler = LambdaLR(optimizer, lr_lambda)
start_momentum = 0.99

/usr/local/lib/python3.11/dist-packages/notebook/utils.py:280: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  return LooseVersion(v) >= LooseVersion(check)
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jchencxh to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Output()

/usr/local/lib/python3.11/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

wandb: WARNING Artifacts logged anonymously cannot be claimed and expire after 7 days.


In [ ]:
global_progress = 0

def train_step(index):
    global total_steps, start_momentum, global_progress
    momentum = start_momentum

    idm.train()
    wm.train()
    batch = next(train_iter)

    idm.label(batch)
    wm_global, wm_local = wm.label(batch)

    wm_global_mean = wm_global.mean()
    wm_local_mean = wm_local.mean()


    # Combine losses
    wm_loss = wm_global_mean + wm_local_mean

    loss = wm_loss / 2

    optimizer.zero_grad()  # Reset gradients after updating

    loss.backward()

    grad_norm = torch.nn.utils.clip_grad.clip_grad_norm_(get_unique_params(idm, wm), 0.2)

    optimizer.step()
    wm.update_ema(momentum)

    if index % 250 == 0:
        print(f"Step {index}, wm loss: {wm_loss.item():.6f}")

    del batch
    del loss
    del wm_loss
    wm.zero_grad()
    idm.zero_grad()
    torch.cuda.empty_cache()
    gc.collect()

def test_step():
    # global global_progress
    torch.cuda.empty_cache()
    idm.eval()
    wm.eval()
    # batch = next(test_iter)
    # idm.label(batch)
    # wm_loss = wm.label(batch)
    _, eval_metrics = eval_latent_repr(train_data, idm, wm)
    print(f"TEST eval_metrics: {eval_metrics}")
    # del batch
    # del wm_loss
    wm.zero_grad()
    idm.zero_grad()
    torch.cuda.empty_cache()
    gc.collect()
for step in range(cfg.stage1.steps + 1):
    train_step(step)

    scheduler.step()

    if step % 2000 == 0:
        test_step()


Step 0, wm loss: 4.248335
Total parameter change: 450.32971608638763
TEST eval_metrics: {'train_acc': 0.12083333333333333, 'train_loss': 2.430011542638143, 'test_acc': 0.1278228759765625, 'test_loss': 2.4271964943036437}
Step 250, wm loss: 0.420553
Step 500, wm loss: 0.329660
Step 750, wm loss: 0.163821
Step 1000, wm loss: 0.079322
Step 1250, wm loss: 0.072093
Step 1500, wm loss: 0.075653
Step 1750, wm loss: 0.078240
Step 2000, wm loss: 0.090041
Total parameter change: 848.853134393692
TEST eval_metrics: {'train_acc': 0.13177083333333334, 'train_loss': 2.4181896686553954, 'test_acc': 0.1163330078125, 'test_loss': 2.420894828159362}
Step 2250, wm loss: 0.096462
Step 2500, wm loss: 0.106248
Step 2750, wm loss: 0.103103
Step 3000, wm loss: 0.105142
Step 3250, wm loss: 0.120023
Step 3500, wm loss: 0.116671
Step 3750, wm loss: 0.121304
Step 4000, wm loss: 0.118442
Total parameter change: 1359.2646711841226
TEST eval_metrics: {'train_acc': 0.17239583333333333, 'train_loss': 2.357492049535115